# 2. Entra ID, Authentication Methods & Access Management

This notebook covers the meat of the Entra domain: identity types, authentication methods (MFA, passwordless), Conditional Access, and RBAC.

## Microsoft Entra ID — what it actually is

Entra ID (formerly Azure AD) is Microsoft's cloud-based identity and access management service. It's the **IdP** for:
- Azure portal and Azure resources
- Microsoft 365 (Outlook, Teams, SharePoint)
- Thousands of third-party SaaS apps (Salesforce, ServiceNow, etc.)
- Your own custom applications

Every Azure subscription is associated with an Entra ID **tenant** (directory).

---
## Identity types in Entra ID

| Type | What it is | Example |
|------|-----------|----------|
| **User** | A person (employee, contractor) | alice@contoso.com |
| **Guest user** | External person invited via B2B | bob@fabrikam.com (federated) |
| **Service principal** | An app's identity in the tenant | API-A's service principal |
| **Managed identity** | Azure-managed SP (no secrets) | Container App's identity |
| **Device** | A registered/joined computer | Alice's laptop |
| **Group** | Collection of users/devices | "Marketing team" group |

### Hybrid identity

Most enterprises have on-prem AD *and* Entra ID. **Hybrid identity** means syncing between them:

| Method | How it works |
|--------|--------------|
| **Microsoft Entra Connect** | Syncs users/groups from on-prem AD to Entra ID |
| **Password hash sync (PHS)** | Hash of the on-prem password hash is synced to Entra ID |
| **Pass-through authentication (PTA)** | Entra ID forwards auth requests to on-prem AD in real time |
| **Federation (AD FS)** | On-prem AD FS handles authentication; Entra ID trusts the result |

**Exam tip**: PHS is the simplest and most recommended hybrid option. Federation is the most complex.

---
## App registrations, enterprise apps, service principals, managed identities

Beginners find this confusing because it looks like four names for the same thing. Here's the mental model:

- **App registration** = the *definition* of an app (name, redirect URIs, allowed scopes, secrets). It lives in the tenant where the developer built the app. One per app.
- **Service principal (Enterprise application)** = the *local instance* of an app in **your** tenant. It's what gets role assignments and consent. You get one automatically when an app is registered in your tenant, or when a multi-tenant app is consented to.
- **Managed identity** = a special service principal that Azure **creates and rotates secrets for you** — used by Azure resources (VMs, Functions, Container Apps) to call other Azure resources. No secret to leak.
  - **System-assigned** = tied to the lifecycle of one resource, deleted with it.
  - **User-assigned** = a standalone resource that can be attached to many resources.

### Analogy

Think of the app registration as a *blueprint* and each service principal as a *house built from that blueprint* in a specific neighborhood (tenant). A managed identity is a house where Azure handles the keys for you so you never have to carry them.

**Exam takeaway**: if a question mentions "no secrets in code" or "an Azure resource calling another Azure resource", the answer is almost always **managed identity**.

---
## Authentication methods

Entra ID supports multiple ways to prove your identity:

| Method | Type | Security level |
|--------|------|----------------|
| Password | Something you know | Low (phishable, reusable) |
| SMS/voice OTP | Something you have | Medium (SIM swappable) |
| Authenticator app (push/TOTP) | Something you have | High |
| FIDO2 security key | Something you have | Very high (phishing-resistant) |
| Windows Hello | Something you are | Very high (biometric + device-bound) |
| Certificate-based auth | Something you have | Very high |

### MFA (Multi-Factor Authentication)

MFA requires **two or more** factors from different categories:
- Something you **know** (password)
- Something you **have** (phone, security key)
- Something you **are** (fingerprint, face)

**Exam fact**: MFA blocks 99.9% of identity attacks (Microsoft's published stat).

### Self-Service Password Reset (SSPR)

Lets users reset their own password using pre-registered methods (phone, email, security questions). Reduces helpdesk calls. Can require 1 or 2 methods for reset.

In [1]:
import json

# Simulate MFA evaluation
def evaluate_mfa(factors: list[str]) -> dict:
    categories = {
        'password': 'something you know',
        'security_question': 'something you know',
        'sms_otp': 'something you have',
        'authenticator_push': 'something you have',
        'fido2_key': 'something you have',
        'fingerprint': 'something you are',
        'face_recognition': 'something you are',
    }
    used_categories = set()
    details = []
    for f in factors:
        cat = categories.get(f, 'unknown')
        used_categories.add(cat)
        details.append({'factor': f, 'category': cat})
    
    is_mfa = len(used_categories) >= 2
    return {
        'factors': details,
        'unique_categories': list(used_categories),
        'is_multi_factor': is_mfa,
        'verdict': '✅ MFA satisfied' if is_mfa else '❌ Single factor only'
    }

print('=== Scenario 1: password only ===')
print(json.dumps(evaluate_mfa(['password']), indent=2))

print('\n=== Scenario 2: password + SMS OTP ===')
print(json.dumps(evaluate_mfa(['password', 'sms_otp']), indent=2))

print('\n=== Scenario 3: password + security question (both "know") ===')
print(json.dumps(evaluate_mfa(['password', 'security_question']), indent=2))

print('\n=== Scenario 4: FIDO2 key + fingerprint (passwordless MFA!) ===')
print(json.dumps(evaluate_mfa(['fido2_key', 'fingerprint']), indent=2))

=== Scenario 1: password only ===
{
  "factors": [
    {
      "factor": "password",
      "category": "something you know"
    }
  ],
  "unique_categories": [
    "something you know"
  ],
  "is_multi_factor": false,
  "verdict": "\u274c Single factor only"
}

=== Scenario 2: password + SMS OTP ===
{
  "factors": [
    {
      "factor": "password",
      "category": "something you know"
    },
    {
      "factor": "sms_otp",
      "category": "something you have"
    }
  ],
  "unique_categories": [
    "something you have",
    "something you know"
  ],
  "is_multi_factor": true,
  "verdict": "\u2705 MFA satisfied"
}

=== Scenario 3: password + security question (both "know") ===
{
  "factors": [
    {
      "factor": "password",
      "category": "something you know"
    },
    {
      "factor": "security_question",
      "category": "something you know"
    }
  ],
  "unique_categories": [
    "something you know"
  ],
  "is_multi_factor": false,
  "verdict": "\u274c Single factor o

Notice scenario 3: password + security question = **NOT MFA** because both are "something you know". The exam tests this.

---
## Conditional Access

Conditional Access is the **policy engine** of Zero Trust in Entra ID. It evaluates signals and enforces decisions:

```
IF [conditions met]  →  THEN [grant/block/require MFA]
```

### Signals (conditions)

| Signal | Example |
|--------|---------|
| User/group membership | "All users in Sales group" |
| Location (IP/named location) | "Not from trusted office IPs" |
| Device platform | "iOS devices only" |
| Device compliance | "Must be Intune-compliant" |
| Application | "When accessing SharePoint" |
| Risk level | "Sign-in risk is medium or high" (Identity Protection) |
| Client app | "Legacy authentication clients" |

### Decisions

| Decision | What happens |
|----------|--------------|
| **Block** | Access denied, period |
| **Grant** | Access allowed |
| **Grant + require MFA** | Allowed only if MFA is completed |
| **Grant + require compliant device** | Allowed only from managed devices |
| **Grant + require approved app** | Only certain apps can access the resource |
| **Session controls** | Limited session (e.g., can't download files) |

In [2]:
from dataclasses import dataclass

@dataclass
class SignIn:
    user: str
    group: str
    app: str
    location: str
    device_compliant: bool
    mfa_done: bool
    risk_level: str  # 'none', 'low', 'medium', 'high'

# Conditional Access policies (like you'd configure in the Entra portal)
CA_POLICIES = [
    {
        'name': 'Require MFA for all users',
        'applies_to': lambda s: True,
        'grant': lambda s: 'allow' if s.mfa_done else 'require_mfa',
    },
    {
        'name': 'Block legacy auth',
        'applies_to': lambda s: s.app == 'legacy-smtp',
        'grant': lambda s: 'block',
    },
    {
        'name': 'Require compliant device for admin portal',
        'applies_to': lambda s: s.app == 'azure-portal' and s.group == 'admins',
        'grant': lambda s: 'allow' if s.device_compliant else 'block',
    },
    {
        'name': 'Block high-risk sign-ins',
        'applies_to': lambda s: s.risk_level in ('medium', 'high'),
        'grant': lambda s: 'block' if s.risk_level == 'high' else ('allow' if s.mfa_done else 'require_mfa'),
    },
]

def evaluate_conditional_access(signin: SignIn) -> list:
    results = []
    for policy in CA_POLICIES:
        if policy['applies_to'](signin):
            decision = policy['grant'](signin)
            results.append({'policy': policy['name'], 'decision': decision})
    # Most restrictive wins
    if any(r['decision'] == 'block' for r in results):
        final = '🚫 BLOCKED'
    elif any(r['decision'] == 'require_mfa' for r in results):
        final = '🔐 REQUIRES MFA'
    else:
        final = '✅ ALLOWED'
    return {'policies_evaluated': results, 'final_decision': final}

scenarios = [
    ('Normal sign-in with MFA',
     SignIn('alice', 'users', 'outlook', 'office', True, True, 'none')),
    ('Sign-in without MFA',
     SignIn('alice', 'users', 'outlook', 'office', True, False, 'none')),
    ('Admin accessing portal from non-compliant device',
     SignIn('bob', 'admins', 'azure-portal', 'office', False, True, 'none')),
    ('High-risk sign-in from unknown location',
     SignIn('alice', 'users', 'outlook', 'unknown', True, True, 'high')),
    ('Legacy SMTP client',
     SignIn('alice', 'users', 'legacy-smtp', 'office', True, True, 'none')),
]

for name, signin in scenarios:
    result = evaluate_conditional_access(signin)
    print(f'=== {name} ===')
    for p in result['policies_evaluated']:
        print(f'  📋 {p["policy"]}: {p["decision"]}')
    print(f'  → {result["final_decision"]}\n')

=== Normal sign-in with MFA ===
  📋 Require MFA for all users: allow
  → ✅ ALLOWED

=== Sign-in without MFA ===
  📋 Require MFA for all users: require_mfa
  → 🔐 REQUIRES MFA

=== Admin accessing portal from non-compliant device ===
  📋 Require MFA for all users: allow
  📋 Require compliant device for admin portal: block
  → 🚫 BLOCKED

=== High-risk sign-in from unknown location ===
  📋 Require MFA for all users: allow
  📋 Block high-risk sign-ins: block
  → 🚫 BLOCKED

=== Legacy SMTP client ===
  📋 Require MFA for all users: allow
  📋 Block legacy auth: block
  → 🚫 BLOCKED



---
## Conditional Access — from bad practice to Zero Trust

Let's see the same user, Alice, try to sign in under progressively stronger policies. This is the real bad→best journey most organizations go through.

In [3]:
# Reuse the SignIn + policy engine defined above, but show different *policy sets*.

def run_policies(policies, scenarios):
    for name, si in scenarios:
        results=[]
        for p in policies:
            if p['applies_to'](si):
                results.append((p['name'], p['grant'](si)))
        decisions=[d for _,d in results]
        if 'block' in decisions: final='🚫 BLOCKED'
        elif 'require_mfa' in decisions: final='🔐 REQUIRES MFA'
        else: final='✅ ALLOWED'
        print(f'  {name}: {final}')

risky = SignIn('alice','users','outlook','unknown-country', True, False, 'high')
normal = SignIn('alice','users','outlook','office', True, True, 'none')
scenarios=[('Normal sign-in', normal), ('Risky sign-in (no MFA, high risk)', risky)]

print('❌ Stage 1 — No Conditional Access at all (passwords only)')
run_policies([], scenarios)

print('\n🟡 Stage 2 — Require MFA for everyone (good baseline)')
run_policies([
    {'name':'MFA for all','applies_to':lambda s: True,
     'grant':lambda s: 'allow' if s.mfa_done else 'require_mfa'},
], scenarios)

print('\n🟢 Stage 3 — Zero Trust (MFA + block high risk + block legacy auth)')
run_policies([
    {'name':'MFA for all','applies_to':lambda s: True,
     'grant':lambda s: 'allow' if s.mfa_done else 'require_mfa'},
    {'name':'Block high risk','applies_to':lambda s: s.risk_level=='high',
     'grant':lambda s: 'block'},
    {'name':'Block legacy auth','applies_to':lambda s: s.app=='legacy-smtp',
     'grant':lambda s: 'block'},
], scenarios)

print('\nLesson: each stage raises the floor. A risky sign-in that Stage 1 allows gets MFA-challenged at Stage 2 and outright blocked at Stage 3.')

❌ Stage 1 — No Conditional Access at all (passwords only)
  Normal sign-in: ✅ ALLOWED
  Risky sign-in (no MFA, high risk): ✅ ALLOWED

🟡 Stage 2 — Require MFA for everyone (good baseline)
  Normal sign-in: ✅ ALLOWED
  Risky sign-in (no MFA, high risk): 🔐 REQUIRES MFA

🟢 Stage 3 — Zero Trust (MFA + block high risk + block legacy auth)
  Normal sign-in: ✅ ALLOWED
  Risky sign-in (no MFA, high risk): 🚫 BLOCKED

Lesson: each stage raises the floor. A risky sign-in that Stage 1 allows gets MFA-challenged at Stage 2 and outright blocked at Stage 3.


### Key exam points on Conditional Access

- Requires **Entra ID Premium P1** (at minimum).
- Policies are **additive** — most restrictive wins.
- "Report-only" mode lets you test policies before enforcing them.
- **Named locations** let you trust specific IP ranges (e.g., your office).

---
## RBAC (Role-Based Access Control)

RBAC assigns permissions through **roles** rather than granting access to individual users. Entra ID has built-in roles:

| Role | What it can do |
|------|-----------|
| **Global Administrator** | Full access to everything in the tenant |
| **User Administrator** | Manage users and groups |
| **Security Administrator** | Manage security features (Conditional Access, Identity Protection) |
| **Security Reader** | Read security info but can't change anything |
| **Billing Administrator** | Manage subscriptions and billing |
| **Application Administrator** | Manage app registrations and enterprise apps |

There are also **Azure RBAC roles** for resource access:

| Role | Scope |
|------|-----------|
| **Owner** | Full access + can assign roles |
| **Contributor** | Full access but can't assign roles |
| **Reader** | View only |
| **User Access Administrator** | Manage user access to resources |

### Exam tip

- Entra ID roles → manage the **directory** (users, groups, apps).
- Azure RBAC roles → manage **Azure resources** (VMs, storage, etc.).
- **Least privilege**: always assign the narrowest role that works.

In [4]:
# Simulate RBAC
ROLE_PERMISSIONS = {
    'Owner':       {'read', 'write', 'delete', 'assign_roles'},
    'Contributor': {'read', 'write', 'delete'},
    'Reader':      {'read'},
}

ROLE_ASSIGNMENTS = {
    'alice': {'role': 'Owner',       'scope': '/subscriptions/123/resourceGroups/prod'},
    'bob':   {'role': 'Contributor', 'scope': '/subscriptions/123/resourceGroups/prod'},
    'carol': {'role': 'Reader',      'scope': '/subscriptions/123/resourceGroups/prod'},
}

def check_access(user: str, action: str) -> str:
    assignment = ROLE_ASSIGNMENTS.get(user)
    if not assignment:
        return f'❌ {user}: no role assigned → access denied'
    perms = ROLE_PERMISSIONS[assignment['role']]
    if action in perms:
        return f'✅ {user} ({assignment["role"]}): {action} → allowed'
    return f'❌ {user} ({assignment["role"]}): {action} → denied (not in {assignment["role"]} permissions)'

actions = ['read', 'write', 'delete', 'assign_roles']
for user in ['alice', 'bob', 'carol', 'dave']:
    for action in actions:
        print(check_access(user, action))
    print()

✅ alice (Owner): read → allowed
✅ alice (Owner): write → allowed
✅ alice (Owner): delete → allowed
✅ alice (Owner): assign_roles → allowed

✅ bob (Contributor): read → allowed
✅ bob (Contributor): write → allowed
✅ bob (Contributor): delete → allowed
❌ bob (Contributor): assign_roles → denied (not in Contributor permissions)

✅ carol (Reader): read → allowed
❌ carol (Reader): write → denied (not in Reader permissions)
❌ carol (Reader): delete → denied (not in Reader permissions)
❌ carol (Reader): assign_roles → denied (not in Reader permissions)

❌ dave: no role assigned → access denied
❌ dave: no role assigned → access denied
❌ dave: no role assigned → access denied
❌ dave: no role assigned → access denied



---
## Summary

| Concept | Key fact |
|---------|----------|
| **Identity types** | Users, guests (B2B), service principals, managed identities, devices, groups |
| **Hybrid identity** | Entra Connect syncs on-prem AD → Entra ID. PHS is simplest. |
| **MFA** | ≥2 factors from different categories. Blocks 99.9% of attacks. |
| **SSPR** | Users reset their own passwords. Reduces helpdesk load. |
| **Conditional Access** | IF signals → THEN grant/block/require MFA. Needs Premium P1. |
| **Entra roles** | Manage the directory (Global Admin, User Admin, etc.) |
| **Azure RBAC** | Manage Azure resources (Owner, Contributor, Reader) |

**Next**: [Notebook 3 — Identity Governance](03_identity_governance.ipynb)